In [2]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# 1. Import Data

In [4]:
# Define the file path
# To download original dataset go to - 
# https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

file_path = r"N:\Geodatabase\Raw_Data\Census 2021\Population\RM200 - Sex by single year of age (detailed).csv"

# Read the CSV file
census2021_age_by_single_year_msoa2021_df = pd.read_excel(file_path)

# Display the first few rows
census2021_age_by_single_year_msoa2021_df.head()

ValueError: Excel file format cannot be determined, you must specify an engine manually.

# 2. Data Processing

In [4]:
census2021_age_by_single_year_msoa2021_df_P = pd.pivot_table(census2021_age_by_single_year_msoa2021_df, values='Observation', index=['Middle layer Super Output Areas Code'], columns=['Age (101 categories)'], aggfunc=np.sum)
census2021_age_by_single_year_msoa2021_df_P = census2021_age_by_single_year_msoa2021_df_P.reset_index()

#calculate total ahe per msoa
census2021_age_by_single_year_msoa2021_df_P['total_age_pop'] = census2021_age_by_single_year_msoa2021_df_P.sum(axis=1,numeric_only=True)

#renaming age columns
for i in range(1,101):
    census2021_age_by_single_year_msoa2021_df_P.rename(columns = {'Aged ' + str(i) + ' years':'age_' + str(i)}, inplace = True)
    census2021_age_by_single_year_msoa2021_df_P.rename(columns = {'Aged ' + str(i) + ' year':'age_' + str(i)}, inplace = True)
    census2021_age_by_single_year_msoa2021_df_P.rename(columns = {'Aged ' + str(i) + ' years and over':'age_' + str(i)}, inplace = True)
    census2021_age_by_single_year_msoa2021_df_P.rename(columns = {'Aged under ' + str(i) + ' year':'age_0'}, inplace = True)

#renaming msoa columns
census2021_age_by_single_year_msoa2021_df_P.rename(columns = {'Middle layer Super Output Areas Code':'msoa21cd', 'Age (101 categories)':'fid'}, inplace = True)

census2021_age_by_single_year_msoa2021_df_P.head()

Age (101 categories),msoa21cd,age_1,age_10,age_100,age_11,age_12,age_13,age_14,age_15,age_16,age_17,age_18,age_19,age_2,age_20,age_21,age_22,age_23,age_24,age_25,age_26,age_27,age_28,age_29,age_3,age_30,age_31,age_32,age_33,age_34,age_35,age_36,age_37,age_38,age_39,age_4,age_40,age_41,age_42,age_43,age_44,age_45,age_46,age_47,age_48,age_49,age_5,age_50,age_51,age_52,age_53,age_54,age_55,age_56,age_57,age_58,age_59,age_6,age_60,age_61,age_62,age_63,age_64,age_65,age_66,age_67,age_68,age_69,age_7,age_70,age_71,age_72,age_73,age_74,age_75,age_76,age_77,age_78,age_79,age_8,age_80,age_81,age_82,age_83,age_84,age_85,age_86,age_87,age_88,age_89,age_9,age_90,age_91,age_92,age_93,age_94,age_95,age_96,age_97,age_98,age_99,age_0,total_age_pop
0,E02000001,57,50,3,42,34,22,26,32,36,37,60,52,34,113,131,193,307,221,245,262,240,227,239,39,243,201,211,186,159,174,152,129,134,135,40,113,108,103,117,106,108,96,107,100,133,27,123,138,142,141,96,113,98,111,98,98,28,103,95,84,100,77,87,80,67,75,82,42,58,62,61,76,63,49,51,46,34,28,30,45,42,28,27,22,21,18,14,11,17,33,12,4,3,8,3,3,4,0,0,0,44,8579
1,E02000002,119,149,3,176,147,152,144,124,128,166,125,74,124,66,78,81,88,97,99,101,74,128,118,140,101,139,139,124,164,124,146,141,114,125,156,133,119,104,115,117,95,119,119,106,101,166,100,89,109,88,76,90,85,84,77,75,147,87,53,62,51,64,43,45,46,48,53,140,37,31,45,41,41,36,49,43,34,29,154,36,22,29,30,31,24,21,12,13,20,146,10,10,12,9,7,4,6,3,0,1,115,8281
2,E02000003,176,139,2,155,177,151,147,135,152,133,144,150,180,123,129,146,149,191,129,204,189,192,155,205,142,192,256,190,205,218,196,207,183,153,189,217,179,172,148,132,135,139,145,139,127,151,161,130,144,145,144,143,173,123,97,112,197,137,107,116,104,83,92,79,58,64,65,166,67,65,55,70,56,48,54,41,52,29,177,65,54,30,33,34,28,25,21,15,17,146,13,15,9,4,5,3,6,2,0,0,201,11548
3,E02000004,73,82,2,75,68,99,79,87,97,77,68,63,80,67,71,87,96,102,93,110,97,101,90,88,113,112,96,96,98,85,112,103,90,89,88,94,101,86,72,81,84,80,96,74,81,92,85,91,99,87,93,85,87,78,77,75,84,94,86,75,71,68,62,54,55,45,46,103,36,43,51,47,53,43,25,38,44,34,70,38,29,29,30,25,20,20,15,24,16,95,13,19,10,11,1,6,5,8,1,2,65,6641
4,E02000005,191,185,0,192,200,168,237,162,161,145,141,125,208,116,109,133,107,156,160,129,125,129,148,195,128,161,170,189,214,208,216,206,209,173,220,229,203,173,145,189,157,163,147,135,140,214,141,134,143,134,132,154,120,118,118,93,195,91,75,84,71,57,70,66,45,51,33,201,45,42,39,57,44,36,40,31,24,22,181,27,32,21,24,30,17,20,19,10,8,185,6,5,7,7,1,3,4,0,0,0,162,11086


# 3. Feature Engineering

In [5]:
#Create new age bands

# Define age bands with their corresponding ranges
age_bands = {
    'age_0_to_4': range(0, 5),
    'age_5_to_9': range(5, 10),
    'age_10_to_15': range(10, 16),
    'age_16_to_19': range(16, 20),
    'age_20_to_24': range(20, 25),
    'age_25_to_29': range(25, 30),
    'age_30_to_34': range(30, 35),
    'age_35_to_49': range(35, 50),
    'age_50_to_64': range(50, 65),
    'age_65_to_74': range(65, 75),
    'age_75_to_84': range(75, 85),
    'age_85_to_100': range(85, 101),
}

# Loop through each age band and calculate count & percentage
for band, ages in age_bands.items():
    count_column = f"{band}_count"
    perc_column = f"{band}_perc"

    # Sum age groups within the band
    census2021_age_by_single_year_msoa2021_df_P[count_column] = census2021_age_by_single_year_msoa2021_df_P[
        [f'age_{age}' for age in ages]
    ].sum(axis=1)

    # Calculate percentage of total population
    census2021_age_by_single_year_msoa2021_df_P[perc_column] = np.round(
        (census2021_age_by_single_year_msoa2021_df_P[count_column] /
         census2021_age_by_single_year_msoa2021_df_P['total_age_pop']) * 100, 2
    )

census2021_age_by_single_year_msoa2021_df_P.head()

Age (101 categories),msoa21cd,age_1,age_10,age_100,age_11,age_12,age_13,age_14,age_15,age_16,age_17,age_18,age_19,age_2,age_20,age_21,age_22,age_23,age_24,age_25,age_26,age_27,age_28,age_29,age_3,age_30,age_31,age_32,age_33,age_34,age_35,age_36,age_37,age_38,age_39,age_4,age_40,age_41,age_42,age_43,age_44,age_45,age_46,age_47,age_48,age_49,age_5,age_50,age_51,age_52,age_53,age_54,age_55,age_56,age_57,age_58,age_59,age_6,age_60,age_61,age_62,age_63,age_64,age_65,age_66,age_67,age_68,age_69,age_7,age_70,age_71,age_72,age_73,age_74,age_75,age_76,age_77,age_78,age_79,age_8,age_80,age_81,age_82,age_83,age_84,age_85,age_86,age_87,age_88,age_89,age_9,age_90,age_91,age_92,age_93,age_94,age_95,age_96,age_97,age_98,age_99,age_0,total_age_pop,age_0_to_4_count,age_0_to_4_perc,age_5_to_9_count,age_5_to_9_perc,age_10_to_15_count,age_10_to_15_perc,age_16_to_19_count,age_16_to_19_perc,age_20_to_24_count,age_20_to_24_perc,age_25_to_29_count,age_25_to_29_perc,age_30_to_34_count,age_30_to_34_perc,age_35_to_49_count,age_35_to_49_perc,age_50_to_64_count,age_50_to_64_perc,age_65_to_74_count,age_65_to_74_perc,age_75_to_84_count,age_75_to_84_perc,age_85_to_100_count,age_85_to_100_perc
0,E02000001,57,50,3,42,34,22,26,32,36,37,60,52,34,113,131,193,307,221,245,262,240,227,239,39,243,201,211,186,159,174,152,129,134,135,40,113,108,103,117,106,108,96,107,100,133,27,123,138,142,141,96,113,98,111,98,98,28,103,95,84,100,77,87,80,67,75,82,42,58,62,61,76,63,49,51,46,34,28,30,45,42,28,27,22,21,18,14,11,17,33,12,4,3,8,3,3,4,0,0,0,44,8579,214,2.49,160,1.87,206,2.40,185,2.16,965,11.25,1213,14.14,1000,11.66,1815,21.16,1617,18.85,711,8.29,372,4.34,121,1.41
1,E02000002,119,149,3,176,147,152,144,124,128,166,125,74,124,66,78,81,88,97,99,101,74,128,118,140,101,139,139,124,164,124,146,141,114,125,156,133,119,104,115,117,95,119,119,106,101,166,100,89,109,88,76,90,85,84,77,75,147,87,53,62,51,64,43,45,46,48,53,140,37,31,45,41,41,36,49,43,34,29,154,36,22,29,30,31,24,21,12,13,20,146,10,10,12,9,7,4,6,3,0,1,115,8281,654,7.90,753,9.09,892,10.77,493,5.95,410,4.95,520,6.28,667,8.05,1778,21.47,1190,14.37,430,5.19,339,4.09,155,1.87
2,E02000003,176,139,2,155,177,151,147,135,152,133,144,150,180,123,129,146,149,191,129,204,189,192,155,205,142,192,256,190,205,218,196,207,183,153,189,217,179,172,148,132,135,139,145,139,127,151,161,130,144,145,144,143,173,123,97,112,197,137,107,116,104,83,92,79,58,64,65,166,67,65,55,70,56,48,54,41,52,29,177,65,54,30,33,34,28,25,21,15,17,146,13,15,9,4,5,3,6,2,0,0,201,11548,951,8.24,837,7.25,904,7.83,579,5.01,738,6.39,869,7.53,985,8.53,2490,21.56,1919,16.62,671,5.81,440,3.81,165,1.43
3,E02000004,73,82,2,75,68,99,79,87,97,77,68,63,80,67,71,87,96,102,93,110,97,101,90,88,113,112,96,96,98,85,112,103,90,89,88,94,101,86,72,81,84,80,96,74,81,92,85,91,99,87,93,85,87,78,77,75,84,94,86,75,71,68,62,54,55,45,46,103,36,43,51,47,53,43,25,38,44,34,70,38,29,29,30,25,20,20,15,24,16,95,13,19,10,11,1,6,5,8,1,2,65,6641,394,5.93,444,6.69,490,7.38,305,4.59,423,6.37,491,7.39,515,7.75,1328,20.00,1251,18.84,492,7.41,335,5.04,173,2.61
4,E02000005,191,185,0,192,200,168,237,162,161,145,141,125,208,116,109,133,107,156,160,129,125,129,148,195,128,161,170,189,214,208,216,206,209,173,220,229,203,173,145,189,157,163,147,135,140,214,141,134,143,134,132,154,120,118,118,93,195,91,75,84,71,57,70,66,45,51,33,201,45,42,39,57,44,36,40,31,24,22,181,27,32,21,24,30,17,20,19,10,8,185,6,5,7,7,1,3,4,0,0,0,162,11086,976,8.80,976,8.80,1144,10.32,572,5.16,621,5.60,691,6.23,862,7.78,2693,24.29,1665,15.02,492,4.44,287,2.59,107,0.97


# 4. Data Check 

In [6]:
# Identify all columns ending with '_perc'
perc_columns = [col for col in census2021_age_by_single_year_msoa2021_df_P.columns if col.endswith('_perc')]

# Sum all percentage columns
census2021_age_by_single_year_msoa2021_df_P['total_percentage_check'] = census2021_age_by_single_year_msoa2021_df_P[perc_columns].sum(axis=1)

# Check if the sum is within the range 99.5 - 100.5
census2021_age_by_single_year_msoa2021_df_P['perc_is_correct'] = census2021_age_by_single_year_msoa2021_df_P['total_percentage_check'].between(99.5, 100.5)

# Count and print the number of rows that don't satisfy the condition
incorrect_rows = census2021_age_by_single_year_msoa2021_df_P[~census2021_age_by_single_year_msoa2021_df_P['perc_is_correct']]
num_incorrect_rows = incorrect_rows.shape[0]

# Print the result
print(f"Number of rows where the total percentage is NOT between 99.5 and 100.5: {num_incorrect_rows}")
census2021_age_by_single_year_msoa2021_df_P.head()

Number of rows where the total percentage is NOT between 99.5 and 100.5: 0


Age (101 categories),msoa21cd,age_1,age_10,age_100,age_11,age_12,age_13,age_14,age_15,age_16,age_17,age_18,age_19,age_2,age_20,age_21,age_22,age_23,age_24,age_25,age_26,age_27,age_28,age_29,age_3,age_30,age_31,age_32,age_33,age_34,age_35,age_36,age_37,age_38,age_39,age_4,age_40,age_41,age_42,age_43,age_44,age_45,age_46,age_47,age_48,age_49,age_5,age_50,age_51,age_52,age_53,age_54,age_55,age_56,age_57,age_58,age_59,age_6,age_60,age_61,age_62,age_63,age_64,age_65,age_66,age_67,age_68,age_69,age_7,age_70,age_71,age_72,age_73,age_74,age_75,age_76,age_77,age_78,age_79,age_8,age_80,age_81,age_82,age_83,age_84,age_85,age_86,age_87,age_88,age_89,age_9,age_90,age_91,age_92,age_93,age_94,age_95,age_96,age_97,age_98,age_99,age_0,total_age_pop,age_0_to_4_count,age_0_to_4_perc,age_5_to_9_count,age_5_to_9_perc,age_10_to_15_count,age_10_to_15_perc,age_16_to_19_count,age_16_to_19_perc,age_20_to_24_count,age_20_to_24_perc,age_25_to_29_count,age_25_to_29_perc,age_30_to_34_count,age_30_to_34_perc,age_35_to_49_count,age_35_to_49_perc,age_50_to_64_count,age_50_to_64_perc,age_65_to_74_count,age_65_to_74_perc,age_75_to_84_count,age_75_to_84_perc,age_85_to_100_count,age_85_to_100_perc,total_percentage_check,perc_is_correct
0,E02000001,57,50,3,42,34,22,26,32,36,37,60,52,34,113,131,193,307,221,245,262,240,227,239,39,243,201,211,186,159,174,152,129,134,135,40,113,108,103,117,106,108,96,107,100,133,27,123,138,142,141,96,113,98,111,98,98,28,103,95,84,100,77,87,80,67,75,82,42,58,62,61,76,63,49,51,46,34,28,30,45,42,28,27,22,21,18,14,11,17,33,12,4,3,8,3,3,4,0,0,0,44,8579,214,2.49,160,1.87,206,2.40,185,2.16,965,11.25,1213,14.14,1000,11.66,1815,21.16,1617,18.85,711,8.29,372,4.34,121,1.41,100.02,True
1,E02000002,119,149,3,176,147,152,144,124,128,166,125,74,124,66,78,81,88,97,99,101,74,128,118,140,101,139,139,124,164,124,146,141,114,125,156,133,119,104,115,117,95,119,119,106,101,166,100,89,109,88,76,90,85,84,77,75,147,87,53,62,51,64,43,45,46,48,53,140,37,31,45,41,41,36,49,43,34,29,154,36,22,29,30,31,24,21,12,13,20,146,10,10,12,9,7,4,6,3,0,1,115,8281,654,7.90,753,9.09,892,10.77,493,5.95,410,4.95,520,6.28,667,8.05,1778,21.47,1190,14.37,430,5.19,339,4.09,155,1.87,99.98,True
2,E02000003,176,139,2,155,177,151,147,135,152,133,144,150,180,123,129,146,149,191,129,204,189,192,155,205,142,192,256,190,205,218,196,207,183,153,189,217,179,172,148,132,135,139,145,139,127,151,161,130,144,145,144,143,173,123,97,112,197,137,107,116,104,83,92,79,58,64,65,166,67,65,55,70,56,48,54,41,52,29,177,65,54,30,33,34,28,25,21,15,17,146,13,15,9,4,5,3,6,2,0,0,201,11548,951,8.24,837,7.25,904,7.83,579,5.01,738,6.39,869,7.53,985,8.53,2490,21.56,1919,16.62,671,5.81,440,3.81,165,1.43,100.01,True
3,E02000004,73,82,2,75,68,99,79,87,97,77,68,63,80,67,71,87,96,102,93,110,97,101,90,88,113,112,96,96,98,85,112,103,90,89,88,94,101,86,72,81,84,80,96,74,81,92,85,91,99,87,93,85,87,78,77,75,84,94,86,75,71,68,62,54,55,45,46,103,36,43,51,47,53,43,25,38,44,34,70,38,29,29,30,25,20,20,15,24,16,95,13,19,10,11,1,6,5,8,1,2,65,6641,394,5.93,444,6.69,490,7.38,305,4.59,423,6.37,491,7.39,515,7.75,1328,20.00,1251,18.84,492,7.41,335,5.04,173,2.61,100.00,True
4,E02000005,191,185,0,192,200,168,237,162,161,145,141,125,208,116,109,133,107,156,160,129,125,129,148,195,128,161,170,189,214,208,216,206,209,173,220,229,203,173,145,189,157,163,147,135,140,214,141,134,143,134,132,154,120,118,118,93,195,91,75,84,71,57,70,66,45,51,33,201,45,42,39,57,44,36,40,31,24,22,181,27,32,21,24,30,17,20,19,10,8,185,6,5,7,7,1,3,4,0,0,0,162,11086,976,8.80,976,8.80,1144,10.32,572,5.16,621,5.60,691,6.23,862,7.78,2693,24.29,1665,15.02,492,4.44,287,2.59,107,0.97,100.00,True


# 5. Import GIS geometry

In [7]:
# Define file paths
server_path = r"N:\Geodatabase\Raw_Data\Census 2021\Boundaries\MSOA_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\MSOA_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"
local_path = r"C:/Users/abhimanya.achara/OneDrive - priorpartners.com/Documents/UK DATA/2021 Census Data/MSOA_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW/MSOA_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

# Attempt to read from the server first, fallback to local path if it fails
try:
    msoa2021boundary_df = gpd.read_file(server_path)
    print("Shapefile loaded successfully from the server.")
except Exception as e:
    print(f"Server path failed, loading from local path. Error: {e}")
    msoa2021boundary_df = gpd.read_file(local_path)

# Display the first few rows
msoa2021boundary_df.head()


Shapefile loaded successfully from the server.


,OBJECTID,MSOA21CD,MSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E02000001,City of London 001,3.150420e+06,9614.398143,c5a1cc9a-af45-4ca1-9560-9cabbe595d0e,"POLYGON ((532153.703 182165.155, 532158.250 18..."
1,2,E02000002,Barking and Dagenham 001,2.161561e+06,8306.888230,c1255e36-08d2-4e39-a111-64e5e35e9cb6,"POLYGON ((548881.304 190819.980, 548881.125 19..."
2,3,E02000003,Barking and Dagenham 002,2.141515e+06,9359.512342,e9fe6914-fe42-4f8b-b1f7-8d4a4b17fc8a,"POLYGON ((548958.555 189072.176, 548954.517 18..."
3,4,E02000004,Barking and Dagenham 003,2.492946e+06,8475.840309,04e09225-5949-426e-a7c0-9015ad43e053,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,5,E02000005,Barking and Dagenham 004,1.187954e+06,7321.657104,3d9714b3-dc08-4ab5-b6fd-54dfe4d5d667,"POLYGON ((549237.051 187627.941, 549241.319 18..."


# 6. GIS data management

### Remove Rename fields

In [8]:
#Drop and rename fields
msoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
msoa2021boundary_df.rename(columns = {'MSOA21CD':'msoa21cd','MSOA21NM':'msoa21nm'}, inplace = True)
msoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_2212\4221660210.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  msoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,OBJECTID,msoa21cd,msoa21nm,geometry
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18..."
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19..."
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18..."
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,5,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18..."


### Combine with boundary lookup table

#### MSOA11 to MSOA21

In [9]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Lookup_for_England_and_Wales.xlsx"

# Read the Excel file
msoalookup_df = pd.read_excel(file_path, sheet_name=0)  # Reads the first sheet by default

#drop unwanted fields
msoalookup_df.drop(['CHGIND','LAD22NMW','ObjectId','MSOA21NM'],1,inplace = True)

# Rename fields
msoalookup_df.rename(
    columns={
        'MSOA11CD': 'msoa11cd',
        'MSOA11NM': 'msoa11nm',
        'MSOA21CD': 'msoa21cd',
        'LAD22CD': 'lad22cd',
        'LAD22NM': 'lad22nm'
    }, 
    inplace=True
)


# Display the first few rows
msoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_2212\3282387155.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  msoalookup_df.drop(['CHGIND','LAD22NMW','ObjectId','MSOA21NM'],1,inplace = True)


,msoa11cd,msoa11nm,msoa21cd,lad22cd,lad22nm
0,E02000001,City of London 001,E02000001,E09000001,City of London
1,E02000002,Barking and Dagenham 001,E02000002,E09000002,Barking and Dagenham
2,E02000003,Barking and Dagenham 002,E02000003,E09000002,Barking and Dagenham
3,E02000004,Barking and Dagenham 003,E02000004,E09000002,Barking and Dagenham
4,E02000005,Barking and Dagenham 004,E02000005,E09000002,Barking and Dagenham


In [10]:
msoa2021boundary_df = pd.merge(msoa2021boundary_df, msoalookup_df, how = 'left', on = 'msoa21cd' )
msoa2021boundary_df.head()

,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham
4,5,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham


#### MSOA21 to WARD22

In [11]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Middle_Layer_Super_Output_Area_(2021)_to_Ward_to_LAD_(December_2022)_Lookup_in_England_and_Wales_v2.csv"

# Read the Excel file
msoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
msoawardlookup_df.drop(['MSOA21NM','MSOA21NMW','WD22NMW','LAD22CD','LAD22NM','LAD22NMW','ObjectId'],1,inplace = True)

# Rename fields
msoawardlookup_df.rename(
    columns={
        'MSOA21CD': 'msoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm'
       }, 
    inplace=True
)

# Display the first few rows
msoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_2212\570579279.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  msoawardlookup_df.drop(['MSOA21NM','MSOA21NMW','WD22NMW','LAD22CD','LAD22NM','LAD22NMW','ObjectId'],1,inplace = True)


,msoa21cd,wd22cd,wd22nm
0,E02001445,E05000949,Park
1,E02001444,E05000950,Ravenmeols
2,E02001451,E05000951,St Oswald
3,E02001452,E05000951,St Oswald
4,E02001446,E05000952,Sudell


In [12]:
msoa2021boundary_df = pd.merge(msoa2021boundary_df, msoawardlookup_df, how = 'left', on = 'msoa21cd')
msoa2021boundary_df.head()

,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Green
4,5,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham,E05014071,Whalebone


#### LAD22 to REGION22

In [13]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_2212\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [14]:
msoa2021boundary_df = pd.merge(msoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
msoa2021boundary_df.head()

,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Green,E12000007,London
4,5,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham,E05014071,Whalebone,E12000007,London


### Add data management fields

In [15]:
# Add new data management fields with specified values
msoa2021boundary_df['source'] = "Census 2021"
msoa2021boundary_df['admin_boundary'] = "MSOA 2021"
msoa2021boundary_df['time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
msoa2021boundary_df['link'] = "https://www.ons.gov.uk/datasets/TS007/editions/2021/versions/3"
msoa2021boundary_df.head()

,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,source,admin_boundary,time_period,link
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Green,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...
4,5,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham,E05014071,Whalebone,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...


### Update Area

In [16]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in msoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    msoa2021boundary_df['area_ha'] = msoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in msoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
msoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,source,admin_boundary,time_period,link,area_ha
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,315.042034
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,216.156066
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,214.151525
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Green,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,249.294610
4,5,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham,E05014071,Whalebone,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,118.795416


# 7. Combine Geometry and data

In [17]:
census2021_age_by_single_year_msoa2021_gdb_df = pd.merge(msoa2021boundary_df,census2021_age_by_single_year_msoa2021_df_P, how = 'left', on='msoa21cd')
census2021_age_by_single_year_msoa2021_gdb_df.head()

,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,source,admin_boundary,time_period,link,area_ha,age_1,age_10,age_100,age_11,age_12,age_13,age_14,age_15,age_16,age_17,age_18,age_19,age_2,age_20,age_21,age_22,age_23,age_24,age_25,age_26,age_27,age_28,age_29,age_3,age_30,age_31,age_32,age_33,age_34,age_35,age_36,age_37,age_38,age_39,age_4,age_40,age_41,age_42,age_43,age_44,age_45,age_46,age_47,age_48,age_49,age_5,age_50,age_51,age_52,age_53,age_54,age_55,age_56,age_57,age_58,age_59,age_6,age_60,age_61,age_62,age_63,age_64,age_65,age_66,age_67,age_68,age_69,age_7,age_70,age_71,age_72,age_73,age_74,age_75,age_76,age_77,age_78,age_79,age_8,age_80,age_81,age_82,age_83,age_84,age_85,age_86,age_87,age_88,age_89,age_9,age_90,age_91,age_92,age_93,age_94,age_95,age_96,age_97,age_98,age_99,age_0,total_age_pop,age_0_to_4_count,age_0_to_4_perc,age_5_to_9_count,age_5_to_9_perc,age_10_to_15_count,age_10_to_15_perc,age_16_to_19_count,age_16_to_19_perc,age_20_to_24_count,age_20_to_24_perc,age_25_to_29_count,age_25_to_29_perc,age_30_to_34_count,age_30_to_34_perc,age_35_to_49_count,age_35_to_49_perc,age_50_to_64_count,age_50_to_64_perc,age_65_to_74_count,age_65_to_74_perc,age_75_to_84_count,age_75_to_84_perc,age_85_to_100_count,age_85_to_100_perc,total_percentage_check,perc_is_correct
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,315.042034,57,50,3,42,34,22,26,32,36,37,60,52,34,113,131,193,307,221,245,262,240,227,239,39,243,201,211,186,159,174,152,129,134,135,40,113,108,103,117,106,108,96,107,100,133,27,123,138,142,141,96,113,98,111,98,98,28,103,95,84,100,77,87,80,67,75,82,42,58,62,61,76,63,49,51,46,34,28,30,45,42,28,27,22,21,18,14,11,17,33,12,4,3,8,3,3,4,0,0,0,44,8579,214,2.49,160,1.87,206,2.40,185,2.16,965,11.25,1213,14.14,1000,11.66,1815,21.16,1617,18.85,711,8.29,372,4.34,121,1.41,100.02,True
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,216.156066,119,149,3,176,147,152,144,124,128,166,125,74,124,66,78,81,88,97,99,101,74,128,118,140,101,139,139,124,164,124,146,141,114,125,156,133,119,104,115,117,95,119,119,106,101,166,100,89,109,88,76,90,85,84,77,75,147,87,53,62,51,64,43,45,46,48,53,140,37,31,45,41,41,36,49,43,34,29,154,36,22,29,30,31,24,21,12,13,20,146,10,10,12,9,7,4,6,3,0,1,115,8281,654,7.90,753,9.09,892,10.77,493,5.95,410,4.95,520,6.28,667,8.05,1778,21.47,1190,14.37,430,5.19,339,4.09,155,1.87,99.98,True
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,214.151525,176,139,2,155,177,151,147,135,152,133,144,150,180,123,129,146,149,191,129,204,189,192,155,205,142,192,256,190,205,218,196,207,183,153,189,217,179,172,148,132,135,139,145,139,127,151,161,130,144,145,144,143,173,123,97,112,197,137,107,116,104,83,92,79,58,64,65,166,67,65,55,70,56,48,54,41,52,29,177,65,54,30,33,34,28,25,21,15,17,146,13,15,9,4,5,3,6,2,0,0,201,11548,951,8.24,837,7.25,904,7.83,579,5.01,738,6.39,869,7.53,985,8.53,2490,21.56,1919,16.62,671,5.81,440,3.81,165,1.43,100.01,True
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Green,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,249.294610,73,82,2,75,68,99,79,87,97,77,68,63,80,67,71,87,96,102,93,110,97,101,90,88,113,112,96,9

# 8. Create dominant field

In [18]:
import numpy as np

# Identify all columns ending with '_perc'
perc_columns = [col for col in census2021_age_by_single_year_msoa2021_gdb_df.columns if col.endswith('_perc')]

# Convert percentage columns to a NumPy array for efficiency
perc_values = census2021_age_by_single_year_msoa2021_gdb_df[perc_columns].values

# Sort each row in descending order & get corresponding column indices
sorted_indices = np.argsort(-perc_values, axis=1)  # Negate for descending sort
sorted_perc_values = np.take_along_axis(perc_values, sorted_indices, axis=1)  # Apply sorting

# Get the column names in the sorted order for each row
sorted_col_names = np.array(perc_columns)[sorted_indices]

# Define dominance threshold
dominance_threshold = 5.0

# Store results
dominant_age_bands = []
dominant_values = []

# Iterate through rows efficiently
for row_idx in range(sorted_perc_values.shape[0]):
    row_values = sorted_perc_values[row_idx]
    row_columns = sorted_col_names[row_idx]
    
    # Always include the first (highest) column
    selected_columns = [row_columns[0].replace('_perc', '')]
    selected_values = [row_values[0]]

    # Compare with the next values to check for a 5% drop
    for i in range(1, len(row_values)):
        if row_values[i - 1] - row_values[i] < dominance_threshold:
            selected_columns.append(row_columns[i].replace('_perc', ''))
            selected_values.append(row_values[i])
        else:
            break  # Stop as soon as the drop is 5% or more

    # Store results
    dominant_age_bands.append(", ".join(selected_columns))  # Join multiple values as a string
    dominant_values.append(", ".join(map(str, selected_values)))  # Store corresponding values

# Assign to dataframe
census2021_age_by_single_year_msoa2021_gdb_df['dominant_age_band'] = dominant_age_bands
census2021_age_by_single_year_msoa2021_gdb_df['dominant_value'] = dominant_values

# Display a sample of the dataframe to verify
census2021_age_by_single_year_msoa2021_gdb_df.head()

,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,source,admin_boundary,time_period,link,area_ha,age_1,age_10,age_100,age_11,age_12,age_13,age_14,age_15,age_16,age_17,age_18,age_19,age_2,age_20,age_21,age_22,age_23,age_24,age_25,age_26,age_27,age_28,age_29,age_3,age_30,age_31,age_32,age_33,age_34,age_35,age_36,age_37,age_38,age_39,age_4,age_40,age_41,age_42,age_43,age_44,age_45,age_46,age_47,age_48,age_49,age_5,age_50,age_51,age_52,age_53,age_54,age_55,age_56,age_57,age_58,age_59,age_6,age_60,age_61,age_62,age_63,age_64,age_65,age_66,age_67,age_68,age_69,age_7,age_70,age_71,age_72,age_73,age_74,age_75,age_76,age_77,age_78,age_79,age_8,age_80,age_81,age_82,age_83,age_84,age_85,age_86,age_87,age_88,age_89,age_9,age_90,age_91,age_92,age_93,age_94,age_95,age_96,age_97,age_98,age_99,age_0,total_age_pop,age_0_to_4_count,age_0_to_4_perc,age_5_to_9_count,age_5_to_9_perc,age_10_to_15_count,age_10_to_15_perc,age_16_to_19_count,age_16_to_19_perc,age_20_to_24_count,age_20_to_24_perc,age_25_to_29_count,age_25_to_29_perc,age_30_to_34_count,age_30_to_34_perc,age_35_to_49_count,age_35_to_49_perc,age_50_to_64_count,age_50_to_64_perc,age_65_to_74_count,age_65_to_74_perc,age_75_to_84_count,age_75_to_84_perc,age_85_to_100_count,age_85_to_100_perc,total_percentage_check,perc_is_correct,dominant_age_band,dominant_value
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,315.042034,57,50,3,42,34,22,26,32,36,37,60,52,34,113,131,193,307,221,245,262,240,227,239,39,243,201,211,186,159,174,152,129,134,135,40,113,108,103,117,106,108,96,107,100,133,27,123,138,142,141,96,113,98,111,98,98,28,103,95,84,100,77,87,80,67,75,82,42,58,62,61,76,63,49,51,46,34,28,30,45,42,28,27,22,21,18,14,11,17,33,12,4,3,8,3,3,4,0,0,0,44,8579,214,2.49,160,1.87,206,2.40,185,2.16,965,11.25,1213,14.14,1000,11.66,1815,21.16,1617,18.85,711,8.29,372,4.34,121,1.41,100.02,True,"age_35_to_49, age_50_to_64, age_25_to_29, age_...","21.16, 18.85, 14.14, 11.66, 11.25, 8.29, 4.34,..."
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,216.156066,119,149,3,176,147,152,144,124,128,166,125,74,124,66,78,81,88,97,99,101,74,128,118,140,101,139,139,124,164,124,146,141,114,125,156,133,119,104,115,117,95,119,119,106,101,166,100,89,109,88,76,90,85,84,77,75,147,87,53,62,51,64,43,45,46,48,53,140,37,31,45,41,41,36,49,43,34,29,154,36,22,29,30,31,24,21,12,13,20,146,10,10,12,9,7,4,6,3,0,1,115,8281,654,7.90,753,9.09,892,10.77,493,5.95,410,4.95,520,6.28,667,8.05,1778,21.47,1190,14.37,430,5.19,339,4.09,155,1.87,99.98,True,age_35_to_49,21.47
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,214.151525,176,139,2,155,177,151,147,135,152,133,144,150,180,123,129,146,149,191,129,204,189,192,155,205,142,192,256,190,205,218,196,207,183,153,189,217,179,172,148,132,135,139,145,139,127,151,161,130,144,145,144,143,173,123,97,112,197,137,107,116,104,83,92,79,58,64,65,166,67,65,55,70,56,48,54,41,52,29,177,65,54,30,33,34,28,25,21,15,17,146,13,15,9,4,5,3,6,2,0,0,201,11548,951,8.24,837,7.25,904,7.83,579,5.01,738,6.39,869,7.53,985,8.53,2490,21.56,1919,16.62,671,5.81,440,3.81,165,1.43,100.01,True,"age_35_to_49, age_50_to_64","21.56, 16.62"
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Gre

# Special Case: Find and remove Duplicates

In [21]:
def remove_duplicates(df, unique_column):
    """
    Removes duplicate rows based on a specified unique column while keeping the first occurrence.
    
    Parameters:
        df (GeoDataFrame or DataFrame): The input data to clean.
        unique_column (str): The column to check for duplicates.
    
    Returns:
        GeoDataFrame or DataFrame: A cleaned dataframe with duplicates removed.
    """
    # Check for duplicates
    duplicate_count = df.duplicated(subset=[unique_column], keep='first').sum()
    
    if duplicate_count > 0:
        print(f"Found {duplicate_count} duplicate(s) in column '{unique_column}'. Removing duplicates...")

        # Remove duplicates, keeping the first occurrence
        df = df.drop_duplicates(subset=[unique_column], keep='first')

        print(f"Duplicates removed. Now {unique_column} has only unique values.")

    else:
        print(f"No duplicates found in column '{unique_column}'. No changes made.")

    return df

# Specify the column where duplicates should be checked
unique_column = "msoa21cd"

# Remove duplicates before uploading to PostGIS
census2021_age_by_single_year_msoa2021_gdb_df = remove_duplicates(census2021_age_by_single_year_msoa2021_gdb_df, unique_column)

census2021_age_by_single_year_msoa2021_gdb_df.head()

Found 23 duplicate(s) in column 'msoa21cd'. Removing duplicates...
Duplicates removed. Now msoa21cd has only unique values.


,OBJECTID,msoa21cd,msoa21nm,geometry,msoa11cd,msoa11nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,source,admin_boundary,time_period,link,area_ha,age_1,age_10,age_100,age_11,age_12,age_13,age_14,age_15,age_16,age_17,age_18,age_19,age_2,age_20,age_21,age_22,age_23,age_24,age_25,age_26,age_27,age_28,age_29,age_3,age_30,age_31,age_32,age_33,age_34,age_35,age_36,age_37,age_38,age_39,age_4,age_40,age_41,age_42,age_43,age_44,age_45,age_46,age_47,age_48,age_49,age_5,age_50,age_51,age_52,age_53,age_54,age_55,age_56,age_57,age_58,age_59,age_6,age_60,age_61,age_62,age_63,age_64,age_65,age_66,age_67,age_68,age_69,age_7,age_70,age_71,age_72,age_73,age_74,age_75,age_76,age_77,age_78,age_79,age_8,age_80,age_81,age_82,age_83,age_84,age_85,age_86,age_87,age_88,age_89,age_9,age_90,age_91,age_92,age_93,age_94,age_95,age_96,age_97,age_98,age_99,age_0,total_age_pop,age_0_to_4_count,age_0_to_4_perc,age_5_to_9_count,age_5_to_9_perc,age_10_to_15_count,age_10_to_15_perc,age_16_to_19_count,age_16_to_19_perc,age_20_to_24_count,age_20_to_24_perc,age_25_to_29_count,age_25_to_29_perc,age_30_to_34_count,age_30_to_34_perc,age_35_to_49_count,age_35_to_49_perc,age_50_to_64_count,age_50_to_64_perc,age_65_to_74_count,age_65_to_74_perc,age_75_to_84_count,age_75_to_84_perc,age_85_to_100_count,age_85_to_100_perc,total_percentage_check,perc_is_correct,dominant_age_band,dominant_value
0,1,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,315.042034,57,50,3,42,34,22,26,32,36,37,60,52,34,113,131,193,307,221,245,262,240,227,239,39,243,201,211,186,159,174,152,129,134,135,40,113,108,103,117,106,108,96,107,100,133,27,123,138,142,141,96,113,98,111,98,98,28,103,95,84,100,77,87,80,67,75,82,42,58,62,61,76,63,49,51,46,34,28,30,45,42,28,27,22,21,18,14,11,17,33,12,4,3,8,3,3,4,0,0,0,44,8579,214,2.49,160,1.87,206,2.40,185,2.16,965,11.25,1213,14.14,1000,11.66,1815,21.16,1617,18.85,711,8.29,372,4.34,121,1.41,100.02,True,"age_35_to_49, age_50_to_64, age_25_to_29, age_...","21.16, 18.85, 14.14, 11.66, 11.25, 8.29, 4.34,..."
1,2,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,216.156066,119,149,3,176,147,152,144,124,128,166,125,74,124,66,78,81,88,97,99,101,74,128,118,140,101,139,139,124,164,124,146,141,114,125,156,133,119,104,115,117,95,119,119,106,101,166,100,89,109,88,76,90,85,84,77,75,147,87,53,62,51,64,43,45,46,48,53,140,37,31,45,41,41,36,49,43,34,29,154,36,22,29,30,31,24,21,12,13,20,146,10,10,12,9,7,4,6,3,0,1,115,8281,654,7.90,753,9.09,892,10.77,493,5.95,410,4.95,520,6.28,667,8.05,1778,21.47,1190,14.37,430,5.19,339,4.09,155,1.87,99.98,True,age_35_to_49,21.47
2,3,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,E05014058,Chadwell Heath,E12000007,London,Census 2021,MSOA 2021,2021-02-21,https://www.ons.gov.uk/datasets/TS007/editions...,214.151525,176,139,2,155,177,151,147,135,152,133,144,150,180,123,129,146,149,191,129,204,189,192,155,205,142,192,256,190,205,218,196,207,183,153,189,217,179,172,148,132,135,139,145,139,127,151,161,130,144,145,144,143,173,123,97,112,197,137,107,116,104,83,92,79,58,64,65,166,67,65,55,70,56,48,54,41,52,29,177,65,54,30,33,34,28,25,21,15,17,146,13,15,9,4,5,3,6,2,0,0,201,11548,951,8.24,837,7.25,904,7.83,579,5.01,738,6.39,869,7.53,985,8.53,2490,21.56,1919,16.62,671,5.81,440,3.81,165,1.43,100.01,True,"age_35_to_49, age_50_to_64","21.56, 16.62"
3,4,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,E05014059,Eastbrook & Rush Gre

# 9. Upload to geodatabase

In [22]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_population_age_msoa_mar2021"  # Desired table name
primary_key_column = "msoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [23]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_age_by_single_year_msoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_age_by_single_year_msoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [25]:
# Publish the GeoDataFrame to PostGIS
census2021_age_by_single_year_msoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"GeoDataFrame successfully published to PostGIS: {db_schema}.{table_name}")

Could not set primary key. It may already exist. Error: (psycopg2.errors.UndefinedColumn) column "objectid" of relation "census2021_population_age_msoa_mar2021" does not exist

[SQL: 
        ALTER TABLE uk_new.census2021_population_age_msoa_mar2021
        ADD CONSTRAINT census2021_population_age_msoa_mar2021_pkey PRIMARY KEY (OBJECTID);
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)
Could not create spatial index. It may already exist. Error: (psycopg2.errors.InFailedSqlTransaction) current transaction is aborted, commands ignored until end of transaction block

[SQL: 
        CREATE INDEX census2021_population_age_msoa_mar2021_geom_idx
        ON uk_new.census2021_population_age_msoa_mar2021
        USING GIST (geometry);
    ]
(Background on this error at: https://sqlalche.me/e/20/2j85)
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_population_age_msoa_mar2021
